# Day 09：面向对象进阶（OOP Part 2）—— 继承、多态、魔术方法与 `@property`

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 9 天！我是你的主讲兼 Tech Lead 张工。昨天我们用 `ChatMessage` 封装了消息结构；今天产品经理林晓要求平台同时支持 **OpenAI** 和 **通义千问（Qwen）**，而且未来还要无缝接入 DeepSeek、Claude 等更多模型。如果在业务代码里到处写 `if vendor == 'openai': ... elif vendor == 'qwen': ...`，系统很快就会变得难以维护。今天我们就用面向对象的四大核心利器——**继承、`@property` 属性守卫、魔术方法（`__str__`/`__repr__`/`__call__`）与多态**——亲手打造智枢平台的统一大模型适配器 `BaseLLMModel`！

## 1. 第一步：准备 Day 08 的标准消息对象 `ChatMessage`
在设计模型适配器之前，先明确输入输出契约：所有模型统一接收 `list[ChatMessage]`，并返回一条 `ChatMessage(role='assistant', ...)`。

In [1]:
# 定义合法的对话角色元组，防止拼写错误混入消息流
VALID_ROLES = ('system', 'user', 'assistant', 'tool')

class ChatMessage:
    """智枢平台标准对话消息类"""

    def __init__(self, role: str, content: str) -> None:
        # 1. 校验角色是否在白名单内
        if role not in VALID_ROLES:
            raise ValueError(f'无效角色: {role!r}')
        # 2. 校验内容必须为非空字符串
        if not isinstance(content, str) or not content.strip():
            raise ValueError('content 必须为非空字符串')
        self.role = role
        self.content = content

    def __str__(self) -> str:
        # 面向用户/日志的友好展示
        return f'[{self.role}] {self.content}'

    def __repr__(self) -> str:
        # 面向开发者的精确调试信息
        return f'ChatMessage(role={self.role!r}, content={self.content!r})'

# 测试创建消息并对比 str() 与 repr()
msg = ChatMessage('user', '你好，请介绍一下什么是多态？')
print('str(msg) :', str(msg))
print('repr(msg):', repr(msg))

str(msg) : [user] 你好，请介绍一下什么是多态？
repr(msg): ChatMessage(role='user', content='你好，请介绍一下什么是多态？')


## 2. 第二步：构建基类 `BaseLLMModel` —— `@property` 守卫与魔术方法
在大模型 API 调用中，`temperature`（采样温度）必须在 `[0.0, 2.0]` 之间。如果外部代码直接写 `model.temperature = 5.0`，直到发 HTTP 请求时才报错就太晚了！

我们使用 `@property` 和 `@temperature.setter` 把属性赋值变成一个**带校验的守门员**：
- 读 `model.temperature` 时自动调用 getter；
- 写 `model.temperature = x` 时自动调用 setter，拦截非法类型（包括 `bool`）和越界数值。
- 同时实现 `__str__`、`__repr__` 和 `__call__`，让模型实例像函数一样可以直接 `model('你好')` 调用！

In [2]:
class BaseLLMModel:
    """大模型适配器基类：定义统一契约、参数校验与魔术方法"""

    def __init__(self, model_name: str, temperature: float = 0.7) -> None:
        # 校验模型名称非空
        if not isinstance(model_name, str) or not model_name.strip():
            raise ValueError('model_name 必须为非空字符串')
        self.model_name = model_name.strip()
        # 关键技巧：在 __init__ 中通过 self.temperature 赋值，自动触发 setter 校验！
        self.temperature = temperature

    @property
    def temperature(self) -> float:
        """Getter：返回内部私有属性 _temperature"""
        return self._temperature

    @temperature.setter
    def temperature(self, value: float) -> None:
        """Setter：校验类型与范围 [0.0, 2.0]"""
        # 注意：Python 中 bool 是 int 的子类，必须先显式排除 bool
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f'temperature 必须是数值类型，当前为 {type(value).__name__}')
        temp = float(value)
        if not (0.0 <= temp <= 2.0):
            raise ValueError(f'temperature 必须在 0.0 到 2.0 之间，当前值为 {temp}')
        self._temperature = temp

    def _validate_messages(self, messages: list[ChatMessage]) -> ChatMessage:
        """公共校验逻辑：确保消息列表非空且元素均为 ChatMessage"""
        if not isinstance(messages, list) or len(messages) == 0:
            raise ValueError('messages 必须是非空的 ChatMessage 列表')
        for m in messages:
            if not isinstance(m, ChatMessage):
                raise TypeError('messages 元素必须是 ChatMessage 实例')
        return messages[-1]

    def generate(self, messages: list[ChatMessage]) -> ChatMessage:
        """核心推理接口，留给子类多态重写"""
        self._validate_messages(messages)
        raise NotImplementedError(f'{self.__class__.__name__} 必须重写 generate() 方法')

    def __call__(self, prompt_or_messages: str | list[ChatMessage]) -> ChatMessage:
        """魔术方法：支持 model('你好') 或 model([...]) 直接调用"""
        if isinstance(prompt_or_messages, str):
            messages = [ChatMessage('user', prompt_or_messages)]
        else:
            messages = prompt_or_messages
        return self.generate(messages)

    def __str__(self) -> str:
        return f'[{self.__class__.__name__}] {self.model_name} (temperature={self.temperature})'

    def __repr__(self) -> str:
        return f'{self.__class__.__name__}(model_name={self.model_name!r}, temperature={self.temperature!r})'

# 验证 @property 参数校验是否生效
base = BaseLLMModel('demo-base', temperature=0.5)
print('正常初始化:', base)
try:
    base.temperature = 2.8
except ValueError as err:
    print('成功拦截越界温度:', err)

正常初始化: [BaseLLMModel] demo-base (temperature=0.5)
成功拦截越界温度: temperature 必须在 0.0 到 2.0 之间，当前值为 2.8


## 3. 第三步：继承与多态 —— 实现 `OpenAIModel` 与 `QwenModel`
现在我们让 `OpenAIModel` 和 `QwenModel` 继承 `BaseLLMModel`：
1. 使用 `super().__init__(...)` 复用父类的 `model_name` 和 `temperature` 校验，绝不重复造轮子；
2. 各自添加厂商专属属性（如 OpenAI 的 `api_base`、通义千问的 `enable_search`）；
3. 各自多态重写 `generate(messages)`，返回标准的 `ChatMessage(role='assistant', ...)`。

In [3]:
class OpenAIModel(BaseLLMModel):
    """OpenAI 模型适配器（离线模拟实现）"""

    def __init__(
        self,
        model_name: str = 'gpt-4o-mini',
        temperature: float = 0.7,
        api_base: str = 'https://api.openai.com/v1',
    ) -> None:
        # 调用父类构造方法，复用 model_name 与 temperature 校验
        super().__init__(model_name=model_name, temperature=temperature)
        self.api_base = api_base

    def generate(self, messages: list[ChatMessage]) -> ChatMessage:
        # 复用父类的消息校验辅助方法
        last_msg = self._validate_messages(messages)
        reply = (
            f'[OpenAI:{self.model_name}|temp={self.temperature}] '
            f'已收到({len(messages)}轮上下文)：{last_msg.content}'
        )
        return ChatMessage('assistant', reply)


class QwenModel(BaseLLMModel):
    """通义千问 模型适配器（离线模拟实现）"""

    def __init__(
        self,
        model_name: str = 'qwen-plus',
        temperature: float = 0.7,
        enable_search: bool = False,
    ) -> None:
        super().__init__(model_name=model_name, temperature=temperature)
        self.enable_search = enable_search

    def generate(self, messages: list[ChatMessage]) -> ChatMessage:
        last_msg = self._validate_messages(messages)
        search_tag = '联网增强' if self.enable_search else '纯模型推理'
        reply = (
            f'[Qwen:{self.model_name}|temp={self.temperature}|{search_tag}] '
            f'通义千问已处理：{last_msg.content}'
        )
        return ChatMessage('assistant', reply)

# 验证继承关系
gpt = OpenAIModel(temperature=0.3)
qwen = QwenModel(temperature=0.9, enable_search=True)
print('isinstance(gpt, BaseLLMModel):', isinstance(gpt, BaseLLMModel))
print('isinstance(qwen, BaseLLMModel):', isinstance(qwen, BaseLLMModel))

isinstance(gpt, BaseLLMModel): True
isinstance(qwen, BaseLLMModel): True


## 4. 第四步：体验多态的威力与 `__call__` 语法糖
写一个业务函数 `route_and_ask(model: BaseLLMModel, prompt: str)`：它完全不需要知道传入的是 OpenAI 还是通义千问，只要是 `BaseLLMModel` 的子类，就能直接 `model(prompt)` 完成调用！这就是企业级架构中**面向接口编程**的精髓。

In [4]:
def route_and_ask(model: BaseLLMModel, prompt: str) -> ChatMessage:
    """多态路由函数：同一套业务代码兼容所有厂商模型"""
    print(f'路由目标 -> {model}')
    # 直接把对象当函数调用，底层自动触发 BaseLLMModel.__call__ -> 子类 generate()
    return model(prompt)

models: list[BaseLLMModel] = [gpt, qwen]
for m in models:
    response = route_and_ask(m, '请总结一下智枢平台 Day 09 的核心收获')
    print('模型回复 ->', response)
    print('-' * 60)

路由目标 -> [OpenAIModel] gpt-4o-mini (temperature=0.3)
模型回复 -> [assistant] [OpenAI:gpt-4o-mini|temp=0.3] 已收到(1轮上下文)：请总结一下智枢平台 Day 09 的核心收获
------------------------------------------------------------
路由目标 -> [QwenModel] qwen-plus (temperature=0.9)
模型回复 -> [assistant] [Qwen:qwen-plus|temp=0.9|联网增强] 通义千问已处理：请总结一下智枢平台 Day 09 的核心收获
------------------------------------------------------------
